# Extração BrasilAPI: feriados nacionais
API → Volume `bronze/raw/feriados` → tabela `bronze.feriados`

In [1]:
import io, json
from datetime import datetime

import requests
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound

spark = DatabricksSession.builder.serverless().getOrCreate()
w = WorkspaceClient()

ANO = datetime.now().year
PASTA = "/Volumes/lakehouse_vertice/bronze/raw/feriados"
ARQUIVO = f"{PASTA}/feriados_{ANO}.json"


def arquivos_no_volume(pasta):
    try:
        return {e.path for e in w.files.list_directory_contents(pasta) if not e.is_directory}
    except NotFound:
        return set()

## 1. Extração da API e gravação no Volume (raw)
Um arquivo por ano: se `feriados_{ANO}.json` já existe, a API não é chamada e nada é sobrescrito.

In [2]:
if ARQUIVO in arquivos_no_volume(PASTA):
    print("já extraído, pulando:", ARQUIVO)
else:
    url = f"https://brasilapi.com.br/api/feriados/v1/{ANO}"
    try:
        resposta = requests.get(url, timeout=30)
        resposta.raise_for_status()
        feriados = resposta.json()

        # open()/os.makedirs gravariam no disco local; a Files API do SDK grava direto no Volume
        w.files.create_directory(PASTA)
        conteudo = json.dumps(feriados, ensure_ascii=False).encode("utf-8")
        w.files.upload(ARQUIVO, io.BytesIO(conteudo), overwrite=False)
        print(len(feriados), "feriados gravados em", ARQUIVO)
    except requests.exceptions.ConnectionError:
        print("Erro de conexão: não foi possível acessar a API BrasilAPI.")
        print("O compute serverless não consegue resolver o DNS de brasilapi.com.br.")
        print("Use um cluster clássico com acesso à internet ou peça ao admin para liberar o host.")

14 feriados gravados em /Volumes/lakehouse_vertice/bronze/raw/feriados/feriados_2026.json


## 2. Tabela bronze

In [3]:
spark.sql(f"""
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.feriados
AS SELECT date AS data, name AS feriado, type AS tipo, weekday AS dia_da_semana,
          _metadata.file_name              AS arquivo,
          _metadata.file_modification_time AS coletado_em
FROM read_files('{PASTA}/', format => 'json', multiLine => true)
""")

spark.table("lakehouse_vertice.bronze.feriados").show(truncate=False)

+----------+------------------------+--------+-------------+------------------+-------------------+
|data      |feriado                 |tipo    |dia_da_semana|arquivo           |coletado_em        |
+----------+------------------------+--------+-------------+------------------+-------------------+
|2026-01-01|Confraternização mundial|national|quinta-feira |feriados_2026.json|2026-10-25 03:25:51|
|2026-02-16|Carnaval                |national|segunda-feira|feriados_2026.json|2026-10-25 03:25:51|
|2026-02-17|Carnaval                |national|terça-feira  |feriados_2026.json|2026-10-25 03:25:51|
|2026-04-03|Sexta-feira Santa       |national|sexta-feira  |feriados_2026.json|2026-10-25 03:25:51|
|2026-04-05|Páscoa                  |national|domingo      |feriados_2026.json|2026-10-25 03:25:51|
|2026-04-21|Tiradentes              |national|terça-feira  |feriados_2026.json|2026-10-25 03:25:51|
|2026-05-01|Dia do trabalho         |national|sexta-feira  |feriados_2026.json|2026-10-25 03:25:51|
